In [6]:
data = {"age": "hello"}
print(data["age"])  # prints "hello"
data["age"] = 621
age = int(data["age"])   # crashes
print(age)  # prints 621


hello
621


One Big Benefit in AI

LLMs can generate messy JSON.

Pydantic checks structure before execution.

That means fewer broken tool calls and safer automation.

In [2]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(name="Ajaz", age="35")
print(user)

#name='Ajaz' age=35

user = User(name="Ajaz", age="35")
print(user)

user = User(name="Ajaz", age="thirty five")

name='Ajaz' age=35
name='Ajaz' age=35


ValidationError: 1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty five', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

In [3]:
class Product(BaseModel):
    name: str
    price: float = Field(gt=0)


p = Product(name="Laptop", price=999.99)

p = Product(name="Laptop", price=-50)
ValidationError: Input should be greater than 0

SyntaxError: invalid syntax (1267235379.py, line 9)

In [4]:
class ToolCall(BaseModel):
    tool: str
    query: str

call = ToolCall(tool="search_web", query="latest AI news")

call = ToolCall(tool="search_web")

ValidationError: 1 validation error for ToolCall
query
  Field required [type=missing, input_value={'tool': 'search_web'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [5]:


class Address(BaseModel):
    city: str
    zip_code: int

class Person(BaseModel):
    name: str
    address: Address

p = Person(
    name="Ajaz",
    address={"city": "Dallas", "zip_code": 75001}
)

p = Person(
    name="Ajaz",
    address={"city": "Dallas", "zip_code": "ABC"}
)



ValidationError: 1 validation error for Person
address.zip_code
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='ABC', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

In [6]:
## Example 2 — Pydantic Field Validator (Decorator with Argument)


from pydantic import BaseModel, field_validator


class User(BaseModel):
    name: str
    age: int
    email: str
    password: str

    @field_validator("age")
    @classmethod
    def age_must_be_positive(cls, value):
        if value <= 0:
            raise ValueError("Age must be positive")
        return value

    @field_validator("email")
    @classmethod
    def email_must_have_at(cls, value):
        if "@" not in value:
            raise ValueError("Invalid email address")
        return value

    @field_validator("password")
    @classmethod    
    def password_min_length(cls, value):
        if len(value) < 8:
            raise ValueError("Password must be at least 8 characters")
        return value

In [7]:
# Valid user
user = User(name="Ajaz", age=30, email="info@razsystems.com", password="securepass")
print(user)

# Invalid age
try:
    user = User(name="Ajaz", age=-5, email="info@razsystems.com", password="securepass")
except Exception as e:
    print(e)

# Invalid email
try:
    user = User(name="Ajaz", age=30, email="notanemail", password="securepass")
except Exception as e:
    print(e)

# Weak password
try:
    user = User(name="Ajaz", age=30, email="info@razsystems.com", password="abc")
except Exception as e:
    print(e)


name='Ajaz' age=30 email='info@razsystems.com' password='securepass'
1 validation error for User
age
  Value error, Age must be positive [type=value_error, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error
1 validation error for User
email
  Value error, Invalid email address [type=value_error, input_value='notanemail', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error
1 validation error for User
password
  Value error, Password must be at least 8 characters [type=value_error, input_value='abc', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error


In [ ]:
# Step 1 — Define the decorator once
def login_required(func):
    def wrapper():
        if not is_logged_in:
            print("Access denied. Please login.")
            return
        func()  # only runs if logged in
    return wrapper


In [ ]:
# Step 2 — Apply to any function with @
@login_required
def dashboard():
    print("Welcome to Dashboard")

@login_required
def orders():
    print("Showing Orders")

@login_required
def reports():
    print("Showing Reports")

In [ ]:
# Step 3 — Test
is_logged_in = False
dashboard()   # Access denied. Please login.
orders()      # Access denied. Please login.

is_logged_in = True
dashboard()   # Welcome to Dashboard
orders()      # Showing Orders
reports()     # Showing Reports